In [1]:
import sys

print("Python utilizado pelo notebook:")
print(sys.executable)

Python utilizado pelo notebook:
c:\Users\USER\Documents\estudos\Projetos_Porffolio\Olist-Ecommerce\.venv\Scripts\python.exe


In [2]:

import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [3]:
# Identifica a raiz do projeto
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretório dos CSVs: {RAW_PATH}")


spark = (
    SparkSession.builder
    .appName("OlistDataProfiling")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Versão do Spark: {spark.version}")

Raiz do projeto: c:\Users\USER\Documents\estudos\Projetos_Porffolio\Olist-Ecommerce
Diretório dos CSVs: c:\Users\USER\Documents\estudos\Projetos_Porffolio\Olist-Ecommerce\data\raw


c:\Users\USER\Documents\estudos\Projetos_Porffolio\Olist-Ecommerce\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Versão do Spark: 4.2.0


In [4]:
FILES = {

    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",

}

In [5]:
missing_files = [
    filename
    for filename in FILES.values()
    if not (RAW_PATH / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Arquivos não encontrados em data/raw: {missing_files}"
    )

print("Todos os arquivos foram encontrados.")

Todos os arquivos foram encontrados.


In [6]:
def read_csv(filename: str):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")
        .csv(str(RAW_PATH / filename))
    )


dataframes = {
    table_name: read_csv(filename)
    for table_name, filename in FILES.items()
}

print("Arquivos carregados:", list(dataframes.keys()))

Arquivos carregados: ['customers', 'orders', 'order_items', 'payments', 'products', 'sellers', 'category_translation']


In [7]:
for table_name, dataframe in dataframes.items():
    row_count = dataframe.count()
    column_count = len(dataframe.columns)

    print(
        f"{table_name}: "
        f"{row_count:,} linhas e {column_count} colunas"
    )

customers: 99,441 linhas e 5 colunas
orders: 99,441 linhas e 8 colunas
order_items: 112,650 linhas e 7 colunas
payments: 103,886 linhas e 5 colunas
products: 32,951 linhas e 9 colunas
sellers: 3,095 linhas e 4 colunas
category_translation: 71 linhas e 2 colunas


In [8]:
dataframes["customers"].printSchema()
dataframes["customers"].show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo  

In [9]:
dataframes["customers"].printSchema()
dataframes["customers"].show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo  

In [10]:
dataframes["orders"].printSchema()
dataframes["orders"].show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [11]:
dataframes["orders"].printSchema()
dataframes["orders"].show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [12]:
dataframes["orders"].printSchema()
dataframes["orders"].show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [13]:
dataframes["payments"].printSchema()
dataframes["payments"].show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.78       |
|42fdf880ba16b47b59251dd489d4441a|1            

In [14]:
dataframes["products"].printSchema()
dataframes["products"].show(5, truncate=False)


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+--------------

In [15]:
dataframes["sellers"].printSchema()
dataframes["sellers"].show(5, truncate=False)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+----------------------+-----------------+------------+
only showi

In [16]:
dataframes["category_translation"].printSchema()
dataframes["category_translation"].show(5, truncate=False)

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
+----------------------+-----------------------------+
only showing top 5 rows


In [17]:
def show_null_profile(table_name, dataframe):
    total_rows = dataframe.count()

    expressions = [
        F.sum(
            F.when(
                F.col(column).isNull()
                | (F.trim(F.col(column).cast("string")) == ""),
                1,
            ).otherwise(0)
        ).alias(column)
        for column in dataframe.columns
    ]

    result = dataframe.select(expressions).first().asDict()

    rows = [
        (
            column,
            null_count,
            round((null_count / total_rows) * 100, 2),
        )
        for column, null_count in result.items()
        if null_count > 0
    ]

    print(f"Tabela: {table_name}")
    print(f"Total de registros: {total_rows:,}")

    if not rows:
        print("Nenhum valor nulo ou vazio encontrado.")
        return

    (
        spark.createDataFrame(
            rows,
            ["column_name", "null_count", "null_percentage"],
        )
        .orderBy(F.desc("null_count"))
        .show(100, truncate=False)
    )
    

In [18]:
for table_name, dataframe in dataframes.items():
    show_null_profile(table_name, dataframe)

Tabela: customers
Total de registros: 99,441
Nenhum valor nulo ou vazio encontrado.
Tabela: orders
Total de registros: 99,441
+-----------------------------+----------+---------------+
|column_name                  |null_count|null_percentage|
+-----------------------------+----------+---------------+
|order_delivered_customer_date|2965      |2.98           |
|order_delivered_carrier_date |1783      |1.79           |
|order_approved_at            |160       |0.16           |
+-----------------------------+----------+---------------+

Tabela: order_items
Total de registros: 112,650
Nenhum valor nulo ou vazio encontrado.
Tabela: payments
Total de registros: 103,886
Nenhum valor nulo ou vazio encontrado.
Tabela: products
Total de registros: 32,951
+--------------------------+----------+---------------+
|column_name               |null_count|null_percentage|
+--------------------------+----------+---------------+
|product_category_name     |610       |1.85           |
|product_name_lenght 

In [19]:
expected_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}

In [20]:
spark.stop()